# Retrospective study - what does the edit know?

Interactive companion to `reports/retrospective/`. Regenerate artifacts with `uv run snuffml study --loso` (slow, honest) before running this notebook.

In [ ]:
import pandas as pd
from snuffml import config
from snuffml.eval import metrics, study

out = config.REPORTS_DIR / 'retrospective'
preds = {m: pd.read_parquet(out / f'preds_{m}.parquet') for m in ['logit', 'hgb']}
snap = {m: metrics.snapshot_metrics(p) for m, p in preds.items()}
for m in snap: snap[m]['era'] = study._era_of_row(snap[m]['season'])
{m: metrics.finale_metrics(s).round(3).to_dict() for m, s in snap.items()}

In [ ]:
# how early does the model find the winner? per-era mean winner probability
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 5))
s = snap['logit']
n_eps = s.groupby('season')['episode'].transform('max')
s = s.assign(frac=(s['episode'] / n_eps * 10).round() / 10)
for era, g in s.groupby('era', observed=True):
    g.groupby('frac')['winner_prob'].mean().plot(ax=ax, marker='o', label=era)
ax.legend(); ax.set_xlabel('season progress'); ax.set_ylabel('mean winner probability');

In [ ]:
# what drives the model?
coefs = pd.read_csv(out / 'logit_coefficients.csv')
imp = pd.read_csv(out / 'hgb_permutation_importance.csv')
display(coefs.head(12))
display(imp.head(8))

In [ ]:
# biggest LOSO misses at the finale: decoy edits the model bought
p = preds['logit']
finale = p[p['episode'] == p.groupby('season')['episode'].transform('max')]
top_pick = finale.sort_values('win_prob', ascending=False).groupby('season').head(1)
misses = top_pick[~top_pick['is_winner']][['season', 'castaway', 'win_prob']]
misses.sort_values('win_prob', ascending=False).head(15)